# Paper → Benchmark-Question Pipeline

Implements **Section 4.2** of *Graph-Structured Reinforcement Learning for Scientific Hypothesis Generation in Materials Design* (Pal & Buehler).

```
PDF papers ──► Marker (PDF→Markdown)
           ──► gpt-4o-mini  (structured section extraction: title, DOI, abstract, results, discussion, conclusion)
           ──► consolidate  (one JSONL record per paper)
           ──► gpt-5.4 high (one open-ended question per paper, tagged with one of 5 reasoning categories)
           ──► gpt-5.4 med  (readability refinement, preserves causal structure)
           ──► benchmark_questions.jsonl
```

Drop your PDFs in `data/corpus/raw_papers/` and run the cells top-to-bottom.

## 0. Imports & paths

In [ ]:
import os, re, json
from pathlib import Path
from typing import Optional, Dict, Any, List

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Load API key from repo-root .env
load_dotenv(Path("../.env"))
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

REPO_ROOT     = Path("..").resolve()
RAW_PDF_DIR   = REPO_ROOT / "data" / "corpus" / "raw_papers"      # input PDFs
MARKER_DIR    = REPO_ROOT / "data" / "corpus" / "marker_runs"     # per-paper markdown + cleaned JSON
CONSOLIDATED  = REPO_ROOT / "data" / "corpus" / "consolidated_papers.json"
QUESTIONS_OUT = REPO_ROOT / "data" / "benchmark" / "benchmark_questions.jsonl"

MARKER_DIR.mkdir(parents=True, exist_ok=True)
QUESTIONS_OUT.parent.mkdir(parents=True, exist_ok=True)

## 1. Step 1 — PDF → Markdown with Marker

Marker performs layout-aware PDF text extraction without OCR or LLM rewrite (paper Ref. [39]). Heavy models are loaded once and reused for every PDF.

In [ ]:
from marker.config.parser   import ConfigParser
from marker.converters.pdf    import PdfConverter
from marker.models            import create_model_dict
from marker.output            import text_from_rendered

# Load Marker model artifacts once (slow on first call)
MARKER_MODELS = create_model_dict()

def pdf_to_markdown(pdf_path: Path, out_dir: Path) -> Path:
    """Convert one PDF to Markdown. Skips if already converted."""
    out_dir.mkdir(parents=True, exist_ok=True)
    out_md = out_dir / (pdf_path.stem + ".md")
    if out_md.exists():
        return out_md

    config    = ConfigParser({})
    converter = PdfConverter(
        config=config.generate_config_dict(),
        artifact_dict=MARKER_MODELS,
    )
    text, _, _ = text_from_rendered(converter(str(pdf_path)))
    out_md.write_text(text, encoding="utf-8")
    return out_md


def convert_all_pdfs(raw_dir: Path = RAW_PDF_DIR, marker_dir: Path = MARKER_DIR) -> None:
    """Convert every PDF in raw_dir → marker_dir/paper_XX/paper_XX.md"""
    pdfs = sorted(raw_dir.glob("*.pdf"))
    for i, pdf in enumerate(tqdm(pdfs, desc="Marker"), start=1):
        pdf_to_markdown(pdf, marker_dir / f"paper_{i:02d}")


convert_all_pdfs()

## 2. Step 2 — Structured section extraction with gpt-4o-mini

Per §4.2: extract `title`, `doi`, `abstract`, `results`, `discussion`, `conclusion` (Introduction / Methods / References are intentionally excluded — they carry background and citations rather than mechanistic findings). The Pydantic schema is enforced via `responses.parse(...)`, so the model output is validated automatically.

In [ ]:
SYSTEM_PROMPT = """You extract specific sections from research papers provided as Markdown text.
Return ONLY the requested JSON object that matches the given schema (no extra keys, no prose)."""

USER_PROMPT_TEMPLATE = """You are given Markdown extracted from a research paper.

Task:
1) Extract:
   - title
   - DOI (if present)
   - abstract
   - results
   - discussion
   - conclusion

2) Robustness rules:
   - Section headings may vary (e.g., "Results", "Results and Discussion", "Findings", "General Discussion",
     "Concluding Remarks", "Summary", "Discussion and Outlook", etc.).
   - If the paper has "Results and Discussion" combined, put the full combined text under "results"
     and set "discussion" to "" (empty string) unless there is a separate Discussion section elsewhere.
   - If "Discussion" exists separately, put it under "discussion".
   - If "Conclusion(s)" does not exist, use the best equivalent (e.g., "Concluding Remarks" or final "Summary/Outlook")
     and note that decision in notes.
   - Prefer verbatim text from the Markdown. Preserve meaningful inline Markdown formatting.
   - Do NOT invent content. If a field is not present, set it to "" (or null for DOI) and add its name to missing_fields.
   - DOI should look like 10.<digits>/<suffix> if possible; if you see a DOI URL, extract the DOI part.

3) Output:
   - JSON exactly matching the schema.
   - Include brief notes for any non-trivial mapping decisions (e.g., used "Outlook" as conclusion).

Markdown:
<<<PAPER_MARKDOWN
{md}
PAPER_MARKDOWN>>>
"""

In [ ]:
"""You extract specific sections from research papers provided as Markdown text.
Return ONLY the requested JSON object that matches the given schema (no extra keys, no prose).
You are given Markdown extracted from a research paper.

Task:
1) Extract:
   - title
   - DOI (if present)
   - abstract
   - results
   - discussion
   - conclusion

2) Robustness rules:
   - Section headings may vary (e.g., "Results", "Results and Discussion", "Findings", "General Discussion",
     "Concluding Remarks", "Summary", "Discussion and Outlook", etc.).
   - If the paper has "Results and Discussion" combined, put the full combined text under "results"
     and set "discussion" to "" (empty string) unless there is a separate Discussion section elsewhere.
   - If "Discussion" exists separately, put it under "discussion".
   - If "Conclusion(s)" does not exist, use the best equivalent (e.g., "Concluding Remarks" or final "Summary/Outlook")
     and note that decision in notes.
   - Prefer verbatim text from the Markdown. Preserve meaningful inline Markdown formatting.
   - Do NOT invent content. If a field is not present, set it to "" (or null for DOI) and add its name to missing_fields.
   - DOI should look like 10.<digits>/<suffix> if possible; if you see a DOI URL, extract the DOI part.

3) Output:
   - JSON exactly matching the schema.
   - Include brief notes for any non-trivial mapping decisions (e.g., used "Outlook" as conclusion).
"""

In [ ]:
class PaperExtract(BaseModel):
    """Schema enforced via the OpenAI Responses API structured-output mode."""
    title:      str
    doi:        Optional[str]
    abstract:   str
    results:    str
    discussion: str
    conclusion: str

EXTRACT_SYSTEM = (
   """You are given Markdown extracted from a research paper.
    Task:
    1) Extract:
    - title
    - DOI (if present)
    - abstract
    - results
    - discussion
    - conclusion

    2) Robustness rules:
    - Section headings may vary (e.g., "Results", "Results and Discussion", "Findings", "General Discussion",
        "Concluding Remarks", "Summary", "Discussion and Outlook", etc.).
    - If the paper has "Results and Discussion" combined, put the full combined text under "results"
        and set "discussion" to "" (empty string) unless there is a separate Discussion section elsewhere.
    - If "Discussion" exists separately, put it under "discussion".
    - If "Conclusion(s)" does not exist, use the best equivalent (e.g., "Concluding Remarks" or final "Summary/Outlook")
        and note that decision in notes.
    - Prefer verbatim text from the Markdown. Preserve meaningful inline Markdown formatting.
    - Do NOT invent content. If a field is not present, set it to "" (or null for DOI) and add its name to missing_fields.
    - DOI should look like 10.<digits>/<suffix> if possible; if you see a DOI URL, extract the DOI part.

    3) Output:
    - JSON exactly matching the schema.
    - Include brief notes for any non-trivial mapping decisions (e.g., used "Outlook" as conclusion)."""
    )

def extract_paper_fields(md_text: str) -> Dict[str, Any]:
    """Run gpt-4o-mini on one paper's markdown; return validated dict."""
    resp = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": EXTRACT_SYSTEM},
            {"role": "user",   "content": md_text},
        ],
        text_format=PaperExtract,
    )
    return resp.output_parsed.model_dump()


def run_metadata_extraction(marker_dir: Path = MARKER_DIR) -> None:
    """Write paper_XX_cleaned.json next to each paper_XX.md (idempotent)."""
    for paper_dir in tqdm(sorted(p for p in marker_dir.iterdir() if p.is_dir()),
                          desc="Extract"):
        md_files = list(paper_dir.glob("*.md"))
        if not md_files:
            continue
        out_json = paper_dir / (md_files[0].stem + "_cleaned.json")
        if out_json.exists():
            continue
        try:
            fields = extract_paper_fields(md_files[0].read_text(encoding="utf-8"))
            out_json.write_text(json.dumps(fields, indent=2, ensure_ascii=False),
                                encoding="utf-8")
        except Exception as e:
            print(f"[error] {paper_dir.name}: {e}")


run_metadata_extraction()

## 3. Step 3 — Consolidate corpus

Merge `abstract + results + discussion + conclusion` into a single text block per paper and write a JSONL of `{title, doi, text, source_path}` records — the input to the question-generation step.

In [ ]:
def merge_paper_text(rec: Dict[str, Any]) -> str:
    """Concatenate the four content sections with simple `## SECTION` headers."""
    parts = []
    for sec in ("abstract", "results", "discussion", "conclusion"):
        body = (rec.get(sec) or "").strip()
        if body:
            parts.append(f"## {sec.upper()}\n{body}")
    return "\n\n".join(parts)


def consolidate_corpus(marker_dir: Path = MARKER_DIR,
                       out_jsonl: Path = CONSOLIDATED) -> List[Dict[str, Any]]:
    """Merge all *_cleaned.json into one JSONL. Skips near-empty extractions."""
    records = []
    for paper_dir in sorted(marker_dir.iterdir()):
        cleaned = list(paper_dir.glob("*_cleaned.json"))
        if not cleaned:
            continue
        rec = json.loads(cleaned[0].read_text(encoding="utf-8"))
        rec["text"]        = merge_paper_text(rec)
        rec["source_path"] = str(cleaned[0].relative_to(REPO_ROOT))
        if len(rec["text"]) >= 200:
            records.append(rec)

    out_jsonl.write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in records),
        encoding="utf-8",
    )
    print(f"Consolidated {len(records)} papers → {out_jsonl.name}")
    return records


corpus = consolidate_corpus()

## 4. Step 4 — Generate one benchmark question per paper (`gpt-5.4`, high effort)

Each paper produces exactly one self-contained, ~150–200 word, research-level question, tagged with one of the **five reasoning categories** defined in §4.2. The Pydantic schema constrains the output; reasoning effort is `high` per the paper.

In [ ]:
system_prompt = """You are an expert in scientific reasoning, multiscale modeling, and mechanistic analysis.

Your task is to generate ONE high-quality evaluation question from a research paper description.

The question must test deep reasoning. It should force reconstruction of mechanisms, not recall of facts.

The model answering the question will NOT see the original paper, so the question must be fully self-contained.

---

GOAL:
Generate one research-level question that requires:
- causal reasoning
- multiple interacting variables
- non-obvious insight (tradeoff, hidden variable, or failure mode)

---

QUESTION TYPE SELECTION:
Randomly select exactly ONE of the following types:

1. causal_multiscale_reasoning
2. tradeoff_and_non_monotonicity
3. hidden_variable_identification
4. model_abstraction_and_breakdown
5. cross_domain_mapping

---

QUESTION REQUIREMENTS:
- Must be a SINGLE paragraph
- Must define a clear system or setup
- Must include specific variables or mechanisms
- Must require constructing a causal or mechanistic model
- Must include a breakdown, paradox, or limitation
- Must end with one of:
  - design strategy
  - hidden variable
  - failure mode
  - explanation of non-equivalence

---

AVOID:
- vague prompts ("explain why")
- purely descriptive questions
- references to the paper or figures
- multi-paragraph or bullet formatting

---

DIFFICULTY TARGET:
A weak model should fail by:
- giving surface-level explanations
- missing hidden variables
- ignoring competing mechanisms

A strong answer should require:
- multiscale reasoning
- interacting mechanisms
- resolving contradictions

---

FRAMING CONSTRAINTS (STRICT):

- The question must be between 150 and 200 words.
- Use no more than 4 sentences before the final task sentence.
- Avoid narrative phrases such as “simulations show,” “it is observed,” or “reminiscent of.”
- Do not interpret the phenomenon (no analogies like binding–unbinding).
- Present observations as direct statements of system behavior.
- The final sentence must begin with:
  "Explain why ..." OR "Then identify ..."
- Avoid stacking more than 2 clauses per sentence.

---

Before outputting, revise the question to:
- remove any unnecessary explanatory phrases
- ensure each variable appears only once unless needed
- ensure the question can be read in one pass without backtracking

Do not include any text outside the JSON.
"""
user_prompt = """You are given:
(1) PAPER TEXT

{text}
"""


In [ ]:
REASONING_CATEGORIES = [
    "causal_multiscale_reasoning",
    "tradeoff_and_non_monotonicity",
    "hidden_variable_identification",
    "model_abstraction_and_breakdown",
    "cross_domain_mapping",
]

class BenchmarkQuestion(BaseModel):
    category: str = Field(description="One of the five reasoning categories")
    question: str = Field(
        description="Single self-contained paragraph, 150-200 words, ending "
                    "with 'Explain why ...' or 'Then identify ...'"
    )

GENERATION_SYSTEM = (
"""You are an expert in scientific reasoning, multiscale modeling, and mechanistic analysis.

Your task is to generate ONE high-quality evaluation question from a research paper description.

The question must test deep reasoning. It should force reconstruction of mechanisms, not recall of facts.

The model answering the question will NOT see the original paper, so the question must be fully self-contained.

---

GOAL:
Generate one research-level question that requires:
- causal reasoning
- multiple interacting variables
- non-obvious insight (tradeoff, hidden variable, or failure mode)

---

QUESTION TYPE SELECTION:
Randomly select exactly ONE of the following types:

1. causal_multiscale_reasoning
2. tradeoff_and_non_monotonicity
3. hidden_variable_identification
4. model_abstraction_and_breakdown
5. cross_domain_mapping

---

QUESTION REQUIREMENTS:
- Must be a SINGLE paragraph
- Must define a clear system or setup
- Must include specific variables or mechanisms
- Must require constructing a causal or mechanistic model
- Must include a breakdown, paradox, or limitation
- Must end with one of:
  - design strategy
  - hidden variable
  - failure mode
  - explanation of non-equivalence

---

AVOID:
- vague prompts ("explain why")
- purely descriptive questions
- references to the paper or figures
- multi-paragraph or bullet formatting

---

DIFFICULTY TARGET:
A weak model should fail by:
- giving surface-level explanations
- missing hidden variables
- ignoring competing mechanisms

A strong answer should require:
- multiscale reasoning
- interacting mechanisms
- resolving contradictions

---

FRAMING CONSTRAINTS (STRICT):

- The question must be between 150 and 200 words.
- Use no more than 4 sentences before the final task sentence.
- Avoid narrative phrases such as “simulations show,” “it is observed,” or “reminiscent of.”
- Do not interpret the phenomenon (no analogies like binding–unbinding).
- Present observations as direct statements of system behavior.
- The final sentence must begin with:
  "Explain why ..." OR "Then identify ..."
- Avoid stacking more than 2 clauses per sentence.

---

Before outputting, revise the question to:
- remove any unnecessary explanatory phrases
- ensure each variable appears only once unless needed
- ensure the question can be read in one pass without backtracking

Do not include any text outside the JSON."""
)

# Safety net: strip any '**Category: X.**' prefix the model emits despite the prompt.
_CAT_PREFIX_RE = re.compile(r"^\s*\*\*Category:\s*[^*]+\*\*\s*")

def generate_question(paper_text: str) -> Optional[BenchmarkQuestion]:
    """One paper → one (category, question) pair, validated against the schema."""
    user = (
        "Generate one research-level evaluation question from the paper text below. "
        f"Choose one category from: {', '.join(REASONING_CATEGORIES)}.\n\n"
        f"Paper text:\n{paper_text[:8_000]}"
    )
    try:
        resp = client.responses.parse(
            model="gpt-5.4",
            instructions=GENERATION_SYSTEM,
            input=user,
            reasoning={"effort": "high"},
            text_format=BenchmarkQuestion,
            max_output_tokens=5_000,
        )
        q = resp.output_parsed
        q.question = _CAT_PREFIX_RE.sub("", q.question, count=1).lstrip()
        return q
    except Exception as e:
        print(f"[error] question generation: {e}")
        return None


def build_benchmark(corpus_path: Path = CONSOLIDATED,
                    out_path: Path  = QUESTIONS_OUT) -> List[Dict[str, Any]]:
    """Generate one question per paper; write JSONL with id, category, question, doi, title."""
    records = [json.loads(l) for l in corpus_path.read_text().splitlines() if l.strip()]
    questions = []
    for i, rec in enumerate(tqdm(records, desc="Generate")):
        q = generate_question(rec["text"])
        if q is None:
            continue
        questions.append({
            "id":       i,
            "title":    rec.get("title", ""),
            "doi":      rec.get("doi"),
            "category": q.category,
            "question": q.question,
        })

    out_path.write_text(
        "\n".join(json.dumps(q, ensure_ascii=False) for q in questions) + "\n",
        encoding="utf-8",
    )
    print(f"Wrote {len(questions)} questions → {out_path}")
    return questions


questions = build_benchmark()

## 5. Step 5 — Refine for readability (`gpt-5.4`, medium effort)

A second pass polishes grammar / clarity while preserving the scientific system, variables, causal structure, question type, and intended reasoning challenge (§4.2). Reads and writes JSONL in place.

In [ ]:
class RefinedQuestion(BaseModel):
    question: str

REFINEMENT_SYSTEM = (
"""
You are a scientific benchmark question editor.

Your task is to rewrite a generated evaluation question to make it clearer, more readable, and more benchmark-ready, without changing its underlying scientific structure.

Preserve:
- the same physical or conceptual system
- the same variables
- the same causal logic
- the same question type
- the same intended hidden variable, tradeoff, failure mode, or breakdown

Improve:
- sentence flow
- readability
- grammar
- precision
- evaluability

Strict rules:
- Do not add new scientific claims.
- Do not remove essential variables.
- Do not change the mechanism being tested.
- Do not make the question easier.
- Do not split into bullet points.
- Output a single paragraph only.
- Keep the rewritten question between 120 and 250 words.
- The final sentence must clearly ask the model to explain a failure, identify a hidden variable, propose a design strategy, or analyze a breakdown.

Return strict JSON only:
{
  "question": "<rewritten single-paragraph question>"
}
"""
)

def refine_question(text: str) -> str:
    """One readability pass; falls back to the original on error."""
    try:
        resp = client.responses.parse(
            model="gpt-5.4",
            instructions=REFINEMENT_SYSTEM,
            input=f"Rewrite this question for readability:\n\n{text}",
            reasoning={"effort": "medium"},
            text_format=RefinedQuestion,
            max_output_tokens=1_000,
        )
        return _CAT_PREFIX_RE.sub("", resp.output_parsed.question, count=1).lstrip()
    except Exception as e:
        print(f"[error] refinement: {e}")
        return text


def refine_benchmark(path: Path = QUESTIONS_OUT) -> None:
    """In-place: keep the original under `question_raw`, replace `question` with refined."""
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    for r in tqdm(rows, desc="Refine"):
        r["question_raw"] = r["question"]
        r["question"]     = refine_question(r["question"])
    path.write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n",
        encoding="utf-8",
    )
    print(f"Refined {len(rows)} questions in place → {path.name}")


refine_benchmark()

## 6. Inspect the final benchmark

In [ ]:
benchmark = pd.read_json(QUESTIONS_OUT, lines=True)
print(f"Benchmark size: {len(benchmark)} questions\n")
print("Category distribution:")
print(benchmark["category"].value_counts(), "\n")
print("Sample:")
sample = benchmark.iloc[0]
print(f"  Title:    {sample['title'][:80]}")
print(f"  Category: {sample['category']}")
print(f"  Question: {sample['question'][:300]}...")